# Whisper-small Fine-Tuning on Children's Speech (Kaggle)

This notebook fine-tunes `openai/whisper-small` on the Pasketti competition data.
Designed for Kaggle T4 GPU (~6 hours training time).

**Requirements:**
- Kaggle dataset with competition audio uploaded as a private dataset
- HuggingFace Hub token (set as Kaggle secret `HF_TOKEN`)
- ~6 hours GPU quota

## 1. Setup

In [ ]:
import subprocess
import sys

# Install dependencies not pre-installed on Kaggle
deps = ["jiwer>=3.0", "audiomentations>=0.36", "peft>=0.12"]
for dep in deps:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path (for Kaggle: upload src/ as a utility script dataset)
# For local dev, this assumes notebook is run from project root
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif Path("/kaggle/input/childwhisper-src/src").exists():
    sys.path.insert(0, "/kaggle/input/childwhisper-src")
    PROJECT_ROOT = Path("/kaggle/input/childwhisper-src")

from src.kaggle_utils import (
    get_paths,
    setup_hub_auth,
    get_latest_checkpoint,
    download_checkpoint,
    get_kaggle_training_args,
    verify_kaggle_data,
    is_kaggle,
)
from src.train_whisper_small import main as train_main

print(f"Running on Kaggle: {is_kaggle()}")
print(f"Project root: {PROJECT_ROOT}")

## 2. Environment & Paths

In [ ]:
# Configure paths based on environment
# On Kaggle: dataset_slug should match your uploaded dataset name
# Locally: point to your data/ directory
DATASET_SLUG = "pasketti-word-audio"  # Change to match your Kaggle dataset
LOCAL_DATA_DIR = "data"  # For local development

paths = get_paths(dataset_slug=DATASET_SLUG, local_data_dir=LOCAL_DATA_DIR)
AUDIO_DIR = paths["audio_dir"]
METADATA_PATH = paths["metadata_path"]
OUTPUT_DIR = paths["output_dir"]

# Config path
CONFIG_PATH = PROJECT_ROOT / "configs" / "training_config.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("configs/training_config.yaml")

print(f"Audio dir:     {AUDIO_DIR}")
print(f"Metadata:      {METADATA_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Config:        {CONFIG_PATH}")

## 3. Data Verification

In [ ]:
stats = verify_kaggle_data(str(AUDIO_DIR), str(METADATA_PATH))

print(f"Utterances:       {stats['num_utterances']}")
print(f"Audio found:      {stats['num_audio_found']}")
print(f"Audio missing:    {stats['num_missing_audio']}")

if stats["duration_stats"]:
    ds = stats["duration_stats"]
    print(f"\nDuration stats:")
    print(f"  Min:          {ds['min']:.2f}s")
    print(f"  Max:          {ds['max']:.2f}s")
    print(f"  Mean:         {ds['mean']:.2f}s")
    print(f"  Total hours:  {ds['total_hours']:.1f}h")

if stats["num_missing_audio"] > 0:
    pct = stats["num_missing_audio"] / stats["num_utterances"] * 100
    print(f"\nWARNING: {pct:.1f}% of audio files are missing!")

## 4. HuggingFace Hub Authentication

In [ ]:
# On Kaggle: add HF_TOKEN as a secret in notebook settings
# Locally: export HF_TOKEN=hf_your_token
HUB_MODEL_ID = "nishantgaurav23/pasketti-whisper-small"
PUSH_TO_HUB = True

try:
    setup_hub_auth()
    print("HF Hub authenticated successfully")
except ValueError as e:
    print(f"Hub auth skipped: {e}")
    print("Checkpoints will be saved locally only")
    PUSH_TO_HUB = False

## 5. Training Config

In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

ws = config.get("whisper_small", {})
common = config.get("common", {})

print("=== Whisper-small Training Config ===")
print(f"Model:            {ws.get('model_name', 'openai/whisper-small')}")
print(f"Learning rate:    {ws.get('learning_rate', 1e-5)}")
print(f"Epochs:           {ws.get('num_train_epochs', 3)}")
print(f"Batch size:       {ws.get('per_device_train_batch_size', 2)}")
print(f"Grad accum:       {ws.get('gradient_accumulation_steps', 8)}")
print(f"Effective batch:  {ws.get('per_device_train_batch_size', 2) * ws.get('gradient_accumulation_steps', 8)}")
print(f"FP16:             {ws.get('fp16', True)}")
print(f"Grad checkpoint:  {ws.get('gradient_checkpointing', True)}")
print(f"SpecAugment:      {common.get('spec_augment', {}).get('apply', True)}")
print(f"Hub model ID:     {ws.get('hub_model_id', 'N/A')}")

## 6. Resume Check

In [ ]:
# Check if there's an existing checkpoint on HF Hub to resume from
RESUME_FROM = None

if PUSH_TO_HUB and get_latest_checkpoint(HUB_MODEL_ID):
    print(f"Found existing checkpoint at {HUB_MODEL_ID}")
    resume_dir = str(OUTPUT_DIR / "resume")
    downloaded = download_checkpoint(HUB_MODEL_ID, resume_dir)
    if downloaded:
        RESUME_FROM = downloaded
        print(f"Downloaded checkpoint to {RESUME_FROM}")
    else:
        print("Failed to download, starting fresh")
else:
    print("No existing checkpoint found, training from scratch")

## 7. Train

In [ ]:
import time

# Build training arguments
train_args = get_kaggle_training_args(
    config_path=str(CONFIG_PATH),
    metadata_path=str(METADATA_PATH),
    audio_dir=str(AUDIO_DIR),
    output_dir=str(OUTPUT_DIR),
    resume_from=RESUME_FROM,
)

print(f"Training args: {' '.join(train_args)}")
print(f"\nStarting training...")

start_time = time.time()
val_wer = train_main(train_args)
elapsed = time.time() - start_time

print(f"\nTraining complete!")
print(f"Validation WER: {val_wer:.4f}")
print(f"Elapsed time: {elapsed / 60:.1f} minutes")

## 8. Evaluate

In [ ]:
print(f"=== Training Summary ===")
print(f"Final Validation WER: {val_wer:.4f}")
print(f"Training time:        {elapsed / 60:.1f} min ({elapsed / 3600:.1f} hrs)")
print(f"Output directory:     {OUTPUT_DIR}")

if val_wer < 0.20:
    print(f"\nTarget WER (<0.20) achieved!")
elif val_wer < 0.25:
    print(f"\nGood progress. Consider more epochs or augmentation.")
else:
    print(f"\nWER still high. Check data quality and training config.")

## 9. Save to HuggingFace Hub

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    api = HfApi()
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        create_pr=False,
    )
    print(f"Model uploaded to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print(f"Hub push disabled. Model saved locally at {OUTPUT_DIR}")
    print("To upload manually: huggingface-cli upload-folder ...")